# Streaming and Auto Loader Examples

This notebook introduces common Databricks streaming patterns:

- Structured Streaming basics
- Auto Loader for incremental file ingestion
- checkpointing and fault tolerance
- writing streams to Delta tables
- simple streaming aggregations

## Why streaming matters

Streaming lets Databricks process continuously arriving data instead of waiting for batch windows. This is useful for event processing, operational analytics, and near real-time pipelines.

In [ ]:
source_path = "/Volumes/main/demo/raw/orders_stream"
checkpoint_path = "/Volumes/main/demo/checkpoints/orders_stream"
target_table = "main.demo.orders_stream_silver"

print({"source_path": source_path, "checkpoint_path": checkpoint_path, "target_table": target_table})

## Auto Loader read pattern

Auto Loader incrementally discovers new files in cloud storage and is a standard Databricks pattern for scalable ingestion.

In [ ]:
raw_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", checkpoint_path + "/schema")
    .load(source_path)
)

raw_stream_df

## Transform the stream

Streaming transformations look similar to batch DataFrame transformations, but they are executed continuously as new data arrives.

In [ ]:
from pyspark.sql import functions as F

transformed_stream_df = (
    raw_stream_df
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("order_amount", F.col("order_amount").cast("double"))
    .filter(F.col("order_id").isNotNull())
)

transformed_stream_df

## Write to a Delta table

Checkpointing is required for reliable streaming execution. Delta is a common sink because it supports downstream analytics and incremental processing.

In [ ]:
silver_query = (
    transformed_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path + "/silver")
    .trigger(availableNow=True)
    .toTable(target_table)
)

## Query the streaming output

After the stream writes to Delta, downstream users can query the target table like any other curated dataset.

In [ ]:
display(spark.sql(f"SELECT * FROM {target_table} ORDER BY ingest_ts DESC"))

## Streaming aggregation example

This shows a simple aggregation pattern for operational monitoring or lightweight near real-time metrics.

In [ ]:
aggregated_stream_df = (
    transformed_stream_df
    .groupBy("customer_region")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("order_amount").alias("total_amount")
    )
)

aggregated_stream_df

## Practical guidance

- Always configure checkpoint locations for production streams
- Use Auto Loader for scalable file discovery instead of naive file listing
- Write raw ingestion first, then transform into silver and gold layers
- Use Delta tables as durable streaming sinks
- Choose trigger modes carefully based on latency and cost requirements